#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim,col

In [0]:
rename_map = {
    'Cid':'customer_number',
    'Cntry':'country'}

#Reading Data From Bronze Layer

In [0]:
df = spark.table('workspace.bronze.erp_loc_a101')

#Data Cleaning

##Trimming String columns

In [0]:
for column in df.dtypes:
    if column[1] == 'string':
        df = df.withColumn(column[0],trim(col(column[0])))

##Substring extraction

In [0]:
df = df.withColumn(
    'CID',
    F.regexp_replace(df['CID'],'-','')
    )


##Standardization

###Value Mapping (e.g. US → United States)


In [0]:
df = df.withColumn(
    'cntry',
    F.when(F.upper(df['CNTRY']).isin('US', 'USA') ,'United States')
    .when(F.upper(df['CNTRY']) == 'DE' , 'Germany')
    .when(F.coalesce(df['CNTRY'],F.lit('')) == '' , 'Unknown')

    .otherwise(df['CNTRY'])
    )



###Column Naming Standardization

In [0]:
for old_name,new_name in rename_map.items():
    df = df.withColumnRenamed(old_name,new_name)

#Loading Data Into Silver Layer

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('silver.erp_location_a101')